For the Eyetracking Experiment the design matrix needs to:
- have balanced L/R target representation per image


In [1]:
from IPython.display import Markdown, display

display(Markdown("""
| | **Neutral L** | **Neutral R** | **Expected L** | **Expected R** | **Unexpected L** | **Unexpected R** |
|---|---|---|---|---|---|---|
| **Short** | 11–18 | 111–118 | 41–48 | 141–148 | 71–78 | 171–178 |
| **Long** | 21–28 | 121–128 | 51–58 | 151–158 | 81–88 | 181–188 |
"""))



| | **Neutral L** | **Neutral R** | **Expected L** | **Expected R** | **Unexpected L** | **Unexpected R** |
|---|---|---|---|---|---|---|
| **Short** | 11–18 | 111–118 | 41–48 | 141–148 | 71–78 | 171–178 |
| **Long** | 21–28 | 121–128 | 51–58 | 151–158 | 81–88 | 181–188 |


In [8]:
def create_cue_dynam(highProb=0.7, lowProb=0.3, neutral=1.0, trials_per_cue=40):
    
   
    trial_per_neutral = trials_per_cue
    
    cue_data = {"cue_names": [".\\cues\\Sea_Animal.png",  ".\\cues\\Water_Vehicle.png",  ".\\cues\\Neutral.png"],
                "cue_highProb_cats": [["dolphin", "whale"], ["speedboat", "submarine"], 
                                    ["dolphin", "whale", "speedboat", "submarine"]],
                
                "cue_lowProb_cats": [["speedboat", "submarine"], ["dolphin", "whale"],
                                    ["dolphin", "whale", "speedboat", "submarine"]]}

    cue_data["cue_highProb"] = []
    cue_data["cue_lowProb"] = []
    
    for cue_id, cue in enumerate(cue_data["cue_names"]):
        if cue != ".\\cues\\Neutral.png":
            cue_data["cue_highProb"].append([np.round(highProb / len(cue_data["cue_highProb_cats"][cue_id]), 2)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(lowProb / len(cue_data["cue_lowProb_cats"][cue_id]), 2)] * len(cue_data["cue_lowProb_cats"][cue_id]))
        
        else:
            cue_data["cue_highProb"].append([np.round(neutral / len(cue_data["cue_highProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(neutral / len(cue_data["cue_lowProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            
    cue_data["high_prob_trials"] = [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_highProb"])]
    
    cue_data["low_prob_trials"] =  [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_lowProb"])]
    
    return cue_data

def assign_trigger(row, late=0.100):

    # ---- MASK ----
    if row['mask_ISI'] == 0.017:
        mask_code = 10
    elif row['mask_ISI'] == late:
        mask_code = 20
    else:
        raise ValueError("Unknown mask type")

    # ---- SIDE ----
    if row['target_loc'] == 'L':
        side_code = 0
    elif row['target_loc'] == 'R':
        side_code = 100
    else:
        raise ValueError("Unknown side")

    # ---- EXPECTATION ----
    if row['expectation'] == 'neutral':
        exp_code = 0
    elif row['expectation'] == 'expected':
        exp_code = 30
    elif row['expectation'] == 'unexpected':
        exp_code = 60
    else:
        raise ValueError("Unknown expectation")

    # ---- IMAGE ----
    image_code = row['image_index']  # 1–8

    return mask_code + side_code + exp_code + image_code

def create_changes(list_length, prec):
    # Number of instances to set as True (2%)
    num_true = int(list_length * prec)

    # Create lists of False values
    catch = [False] * list_length

    # Randomly select indices for fixing and imaging channels
    catch_indices = random.sample(range(list_length), num_true)

    for idx in catch_indices:
        catch[idx] = True
    
    return np.array(catch)

def build_constrained_order(df, rng=None, max_unexpected_run=1, max_attempts=1000):
    
    if rng is None:
        rng = random.Random()

    for _ in range(max_attempts):
        remaining = df.copy().reset_index(drop=True)
        ordered_rows = []
        last_target = None
        unexpected_run = 0
        failed = False

        while len(remaining) > 0:

            # Build valid candidates mask
            valid_mask = np.ones(len(remaining), dtype=bool)

            # Rule 1 — no same target twice in a row
            if last_target is not None:
                valid_mask &= (remaining["target"].astype(str) != str(last_target))

            # Rule 2 — max consecutive unexpected trials
            if unexpected_run >= max_unexpected_run:
                valid_mask &= (remaining["expectation"].astype(str) != "unexpected")

            # Integer positions of valid rows within remaining
            valid_positions = np.where(valid_mask)[0]

            if len(valid_positions) == 0:
                failed = True
                break

            # Pick a random valid position
            chosen_pos = valid_positions[rng.randrange(len(valid_positions))]
            row = remaining.iloc[chosen_pos]

            ordered_rows.append(row)

            # Update state
            last_target = str(row["target"])
            unexpected_run = unexpected_run + 1 if str(row["expectation"]) == "unexpected" else 0

            # Drop by integer position and reset index
            remaining = remaining.drop(remaining.index[chosen_pos]).reset_index(drop=True)

        if not failed:
            return pd.DataFrame(ordered_rows).reset_index(drop=True)

    raise RuntimeError(
        f"Could not build a valid trial order after {max_attempts} attempts. "
        "Consider relaxing the constraints."
    )

def create_block_trials(stim_path, cue_data, random_seed, long_isi=0.1, identity_catch = 1.0, location_catch = 0.0): 
    rng = random.Random(random_seed)
    long_isi = 0.1
    categories = os.listdir(stim_path)
    stimuli = []
    for cat in categories:
        cat_path = os.path.join(stim_path, cat)
        files = os.listdir(cat_path)
        stimuli.extend([f".\\stimuli\\{cat}\\{x}" for x in files])

    # Filter stims
    stimuli = np.array(stimuli)
    stimuli = stimuli[np.argsort(stimuli)]

    data = {"target_id": [],
            "distractor_id": [],
            "distractor_selection_id": [],
            "target": [],
            "distractor": [],
            "distractor_selection":[],
            "distractor_selection_loc":[],
            "target_selection_loc":[],
            "expectation": [],
            "mask_ISI": [],
            "cue": [],
            "target_name": [],
            "target_cat": [],
            "target_loc": []}

    mask_type = [0.017, long_isi]
    distractors = stimuli[np.char.count(stimuli, "mask") > 0]
    distractor_ids = np.arange(len(stimuli))[np.isin(stimuli, distractors)]
    local_distractor_ids = np.arange(0, len(distractors))

    for cue_id, cue in enumerate(cue_data["cue_names"]):
        high_cats = np.array(cue_data["cue_highProb_cats"][cue_id])
        low_cats = np.array(cue_data["cue_lowProb_cats"][cue_id])
        
        if cue !=  '.\\cues\\Neutral.png':
            for i, l_cat in enumerate(low_cats):
                l_cat_stim = stimuli[np.char.count(stimuli, l_cat) > 0]
                targets = l_cat_stim[np.char.count(l_cat_stim, "mask") == 0]
                target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]
            

                trial_collectos = []
                for mask in mask_type:
                
                    h_trials = int(cue_data["low_prob_trials"][cue_id][i])
                    two_images = h_trials // 2
                
                    data["target_id"].extend(np.repeat(target_ids , two_images))
                    data["target"].extend(np.repeat(targets, two_images))
                    data["distractor_selection_id"].extend(np.repeat(np.flip(target_ids), two_images))
                    data["distractor_selection"].extend(np.repeat(np.flip(targets), two_images))

                    
                    data["target_loc"].extend(["L", "R"] * two_images)
                    

                    target_names =[x.split("\\")[-1] for x in targets]
                    target_categories = [x.split("_")[0] for x in target_names]
                    
                    random_distractors = np.random.choice(local_distractor_ids, h_trials)
                    data["distractor_id"].extend([distractor_ids[x] for x in random_distractors])
                    data["distractor"].extend([distractors[x] for x in random_distractors])
                    data["target_name"].extend(np.repeat(target_names , two_images))
                    data["target_cat"].extend(np.repeat(target_categories , two_images))
                    
            
                    data["expectation"].extend(["unexpected"] * h_trials)   
                    data["mask_ISI"].extend([mask] * h_trials)
                    data["cue"].extend([cue] * h_trials)

                    
        for i, h_cat in enumerate(high_cats):
            h_cat_stim = stimuli[np.char.count(stimuli, h_cat) > 0]
            targets = h_cat_stim[np.char.count(h_cat_stim, "mask") == 0]
            target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

            trial_collectos = []
            for mask in mask_type:
                h_trials = int(cue_data["high_prob_trials"][cue_id][i])
                two_images = h_trials // 2
                
                data["target_id"].extend(np.repeat(target_ids , two_images))
                data["target"].extend(np.repeat(targets , two_images))
                data["distractor_selection_id"].extend(np.repeat(np.flip(target_ids), two_images))
                data["distractor_selection"].extend(np.repeat(np.flip(targets), two_images))
                    
                data["target_loc"].extend(["L", "R"] * two_images)
                
                target_names =[x.split("\\")[-1] for x in targets]
                target_categories = [x.split("_")[0] for x in target_names]
                
                random_distractors = np.random.choice(local_distractor_ids, h_trials)
                data["distractor_id"].extend([distractor_ids[x] for x in random_distractors])
                data["distractor"].extend([distractors[x] for x in random_distractors])
                data["target_name"].extend(np.repeat(target_names , two_images))
                data["target_cat"].extend(np.repeat(target_categories , two_images))
                
                
                if cue !=  '.\\cues\\Neutral.png':
                    data["expectation"].extend(["expected"] * h_trials)
                else:
                    data["expectation"].extend(["neutral"]* h_trials)
                    
                data["mask_ISI"].extend([mask] * h_trials)
                data["cue"].extend([cue] * h_trials)
                

                
    data["target_selection_loc"] = [np.random.choice(["L", "R"], 1, p=[0.5, 0.5])[0] for _ in range(len(data["target"]))]
    data["distractor_selection_loc"] = ["R" if x == "L" else "L" for x in data["target_selection_loc"]]
    data["distractor_loc"] = ["R" if x == "L" else "L" for x in data["target_loc"]]

    mapping = { 0: 1,
                1: 2,
                14: 3,
                15: 4,
                6: 5,
                7: 6,
                10: 7,
                11: 8
            }


    for key in data.keys():
        print(f"{key} : {len(data[key])}")
    df = pd.DataFrame(data)
    df['image_index'] = df['target_id'].map(mapping)
    df['trigger'] = df.apply(assign_trigger, axis=1)
    df["identity_catch"] = create_changes(len(df), identity_catch)
    df["location_catch"] = create_changes(len(df), location_catch)

    df = build_constrained_order(df, rng=rng)
    
    return df, stimuli
        

In [15]:
def create_block_trials(stim_path, cue_data, random_seed, long_isi=0.1, identity_catch=1.0, location_catch=0.0):
    rng = random.Random(random_seed)
    long_isi = 0.1
    categories = os.listdir(stim_path)
    stimuli = []
    for cat in categories:
        cat_path = os.path.join(stim_path, cat)
        files = os.listdir(cat_path)
        stimuli.extend([f".\\stimuli\\{cat}\\{x}" for x in files])

    stimuli = np.array(stimuli)
    stimuli = stimuli[np.argsort(stimuli)]

    data = {
        "target_id": [],
        "distractor_id": [],
        "distractor_selection_id": [],
        "target": [],
        "distractor": [],
        "distractor_selection": [],
        "distractor_selection_loc": [],
        "target_selection_loc": [],
        "expectation": [],
        "mask_ISI": [],
        "cue": [],
        "target_name": [],
        "target_cat": [],
        "target_loc": []
    }

    mask_type = [0.017, long_isi]
    distractors = stimuli[np.char.count(stimuli, "mask") > 0]
    distractor_ids = np.arange(len(stimuli))[np.isin(stimuli, distractors)]
    local_distractor_ids = np.arange(0, len(distractors))

    mapping = {
        0: 1,
        1: 2,
        14: 3,
        15: 4,
        6: 5,
        7: 6,
        10: 7,
        11: 8
    }

    def add_trials(targets, target_ids, n_trials, mask, cue, expectation_label):
        """
        Add n_trials rows for a given set of targets, guaranteeing all
        data-dict lists grow by exactly n_trials entries.

        The target images are cycled so each gets as equal exposure as
        possible, independent of whether n_trials is divisible by the
        number of images or by 2.
        """
        n_targets = len(targets)
        if n_targets == 0 or n_trials == 0:
            return

        target_names = [x.split("\\")[-1] for x in targets]
        target_categories = [x.split("_")[0] for x in target_names]

        # Build per-trial target assignments by cycling through images
        # so every image appears as evenly as possible.
        repeats = [n_trials // n_targets + (1 if i < n_trials % n_targets else 0)
                   for i in range(n_targets)]

        trial_target_ids   = []
        trial_targets      = []
        trial_dist_sel_ids = []
        trial_dist_sels    = []
        trial_names        = []
        trial_cats         = []

        for i, r in enumerate(repeats):
            trial_target_ids.extend([target_ids[i]] * r)
            trial_targets.extend([targets[i]] * r)
            # paired distractor-selection is the "other" image (flipped)
            paired_idx = (i + 1) % n_targets
            trial_dist_sel_ids.extend([target_ids[paired_idx]] * r)
            trial_dist_sels.extend([targets[paired_idx]] * r)
            trial_names.extend([target_names[i]] * r)
            trial_cats.extend([target_categories[i]] * r)

        # Interleave L/R target locations as evenly as possible
        locs = (["L", "R"] * (n_trials // 2 + 1))[:n_trials]

        random_distractors = np.random.choice(local_distractor_ids, n_trials)

        data["target_id"].extend(trial_target_ids)
        data["target"].extend(trial_targets)
        data["distractor_selection_id"].extend(trial_dist_sel_ids)
        data["distractor_selection"].extend(trial_dist_sels)
        data["target_loc"].extend(locs)
        data["target_name"].extend(trial_names)
        data["target_cat"].extend(trial_cats)
        data["distractor_id"].extend([distractor_ids[x] for x in random_distractors])
        data["distractor"].extend([distractors[x] for x in random_distractors])
        data["expectation"].extend([expectation_label] * n_trials)
        data["mask_ISI"].extend([mask] * n_trials)
        data["cue"].extend([cue] * n_trials)

    for cue_id, cue in enumerate(cue_data["cue_names"]):
        high_cats = np.array(cue_data["cue_highProb_cats"][cue_id])
        low_cats  = np.array(cue_data["cue_lowProb_cats"][cue_id])
        is_neutral = (cue == '.\\cues\\Neutral.png')

        # ---- LOW-PROBABILITY / UNEXPECTED trials (non-neutral cues only) ----
        if not is_neutral:
            for i, l_cat in enumerate(low_cats):
                l_cat_stim = stimuli[np.char.count(stimuli, l_cat) > 0]
                targets    = l_cat_stim[np.char.count(l_cat_stim, "mask") == 0]
                target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

                for mask in mask_type:
                    # Round to nearest even number so L/R split is always exact
                    n_trials = int(cue_data["low_prob_trials"][cue_id][i])
                    if n_trials % 2 != 0:
                        n_trials += 1
                    add_trials(targets, target_ids, n_trials, mask, cue, "unexpected")

        # ---- HIGH-PROBABILITY / EXPECTED (or NEUTRAL) trials ----
        for i, h_cat in enumerate(high_cats):
            h_cat_stim = stimuli[np.char.count(stimuli, h_cat) > 0]
            targets    = h_cat_stim[np.char.count(h_cat_stim, "mask") == 0]
            target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

            expectation_label = "neutral" if is_neutral else "expected"

            for mask in mask_type:
                n_trials = int(cue_data["high_prob_trials"][cue_id][i])
                if n_trials % 2 != 0:
                    n_trials += 1
                add_trials(targets, target_ids, n_trials, mask, cue, expectation_label)

    # ---- Derived columns (computed after all rows are collected) ----
    n_total = len(data["target"])

    data["target_selection_loc"]    = [np.random.choice(["L", "R"], p=[0.5, 0.5]) for _ in range(n_total)]
    data["distractor_selection_loc"] = ["R" if x == "L" else "L" for x in data["target_selection_loc"]]
    data["distractor_loc"]           = ["R" if x == "L" else "L" for x in data["target_loc"]]

    for key in data.keys():
        print(f"{key} : {len(data[key])}")

    df = pd.DataFrame(data)
    df['image_index'] = df['target_id'].map(mapping)
    df['trigger']     = df.apply(assign_trigger, axis=1)
    df["identity_catch"] = create_changes(len(df), identity_catch)
    df["location_catch"] = create_changes(len(df), location_catch)

    df = build_constrained_order(df, rng=rng)

    return df, stimuli

In [18]:
import numpy as np
import os
import pandas as pd
import random 

cue_data = create_cue_dynam(highProb=0.8, lowProb=0.2, neutral=1.0,trials_per_cue=40)
stim_path = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/01_Eyelink_Task/stimuli"
trials, stimss = create_block_trials(stim_path, cue_data, random_seed=19)

target_id : 240
distractor_id : 240
distractor_selection_id : 240
target : 240
distractor : 240
distractor_selection : 240
distractor_selection_loc : 240
target_selection_loc : 240
expectation : 240
mask_ISI : 240
cue : 240
target_name : 240
target_cat : 240
target_loc : 240
distractor_loc : 240


In [96]:
trials.head(20)

,target_id,distractor_id,distractor_selection_id,target,distractor,distractor_selection,distractor_selection_loc,target_selection_loc,expectation,mask_ISI,cue,target_name,target_cat,target_loc,distractor_loc,image_index,trigger,identity_catch,location_catch
0,15,3,14,.\stimuli\whale\whale_01.jpg,.\stimuli\dolphin\mask-dolphin_01.png,.\stimuli\whale\whale_00.jpg,R,L,expected,0.100,.\cues\Sea_Animal.png,whale_01.jpg,whale,L,R,4,54,True,False
1,1,3,0,.\stimuli\dolphin\dolphin_01.jpg,.\stimuli\dolphin\mask-dolphin_01.png,.\stimuli\dolphin\dolphin_00.jpg,R,L,expected,0.017,.\cues\Sea_Animal.png,dolphin_01.jpg,dolphin,R,L,2,142,True,False
2,14,2,15,.\stimuli\whale\whale_00.jpg,.\stimuli\dolphin\mask-dolphin_00.png,.\stimuli\whale\whale_01.jpg,R,L,neutral,0.017,.\cues\Neutral.png,whale_00.jpg,whale,R,L,3,113,True,False
3,10,13,11,.\stimuli\submarine\submarine_00.jpg,.\stimuli\whale\mask-whale_01.png,.\stimuli\submarine\submarine_01.jpg,L,R,expected,0.017,.\cues\Water_Vehicle.png,submarine_00.jpg,submarine,R,L,7,147,True,False
4,0,8,1,.\stimuli\dolphin\dolphin_00.jpg,.\stimuli\submarine\mask-submarine_00.png,.\stimuli\dolphin\dolphin_01.jpg,L,R,expected,0.100,.\cues\Sea_Animal.png,dolphin_00.jpg,dolphin,R,L,1,151,True,False
5,10,5,11,.\stimuli\submarine\submarine_00.jpg,.\stimuli\speedboat\mask-speedboat_01.png,.\stimuli\submarine\submarine_01.jpg,L,R,unexpected,0.100,.\cues\Sea_Animal.png,submarine_00.jpg,submarine,L,R,7,87,True,False
6,14,12,15,.\stimuli\whale\whale_00.jpg,.\stimuli\whale\mask-whale_00.png,.\stimuli\whale\whale_01.jpg,L,R,neutral,0.017,.\cues\Neutral.png,whale_00.jpg,whale,L,R,3,13,True,False
7,15,8,14,.\stimuli\whale\whale_01.jpg,.\stimuli\submarine\mask-submarine_00.png,.\stimuli\whale\whale_00.jpg,R,L,unexpected,0.100,.\cues\Water_Vehicle.png,whale_01.jpg,whale,R,L,4,184,True,False
8,7,4,6,.\stimuli\speedboat\speedboat_01.jpg,.\stimuli\speedboat\mask-speedboat_00.png,.\stimuli\speedboat\speedboat_00.jpg,R,L,neutral,0.100,.\cues\Neutral.png,speedboat_01.jpg,speedboat,R,L,6,126,True,False
9,6,2,7,.\stimuli\speedboat\speedboat_00.jpg,.\stimuli\dolphin\mask-dolphin_00.png,.\stimuli\speedboat\speedboat_01.jpg,L,R,expected,0.100,.\cues\Water_Vehicle.png,speedboat_00.jpg,speedboat,R,L,5,155,True,False


In [33]:
for key in data.keys():
    print(f"{key}: {len(data[key])}")

target_id: 240
distractor_id: 240
distractor_selection_id: 240
target: 240
distractor: 240
distractor_selection: 240
distractor_selection_loc: 240
expectation: 240
mask_ISI: 240
cue: 240
target_name: 480
target_cat: 480
target_loc: 240
target_selection_loc: 240
distractor_loc: 240


In [85]:
unique_combos = df[['trigger','target_name', 'target_id', 'image_index']].drop_duplicates()
unique_combos.sort_values("trigger")

,trigger,target_name,target_id,image_index
160,11,dolphin_00.jpg,0,1
166,12,dolphin_01.jpg,1,2
180,13,whale_00.jpg,14,3
186,14,whale_01.jpg,15,4
200,15,speedboat_00.jpg,6,5
...,...,...,...,...
101,184,whale_01.jpg,15,4
7,185,speedboat_00.jpg,6,5
9,186,speedboat_01.jpg,7,6
19,187,submarine_00.jpg,10,7


In [1]:
import os
from PIL import Image

stim_dir = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/01_Eyelink_Task/stimuli"
categories = os.listdir(stim_dir)
target_size = (500, 500)

# Image extensions to process
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

for c in categories:
    root_dir = os.path.join(stim_dir, c)
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(valid_exts):
                img_path = os.path.join(root, file)

                try:
                    with Image.open(img_path) as img:
                        img = img.convert("RGB")  # safe for consistency
                        img_resized = img.resize(target_size, Image.LANCZOS)
                        img_resized.save(img_path)

                except Exception as e:
                    print(f"Failed to process {img_path}: {e}")


In [1]:
import math 
def pixels_to_degrees(pixels, distance, screen_width, resolution_width):
    """
    Convert pixels to degrees of visual angle.

    Parameters:
        pixels (int): Number of pixels to convert.
        distance (float): Distance from the observer to the screen (same units as screen_width).
        screen_width (float): Physical width of the screen (same units as distance).
        resolution_width (int): Horizontal resolution of the screen (in pixels).

    Returns:
        float: Visual angle in degrees.
    """
    # Physical size of a single pixel
    pixel_size = screen_width / resolution_width

    # Physical size of the object (in the same units as screen_width)
    object_size = pixel_size * pixels

    # Calculate visual angle using the formula
    visual_angle = 2 * math.degrees(math.atan((object_size / 2) / distance))

    return visual_angle

In [28]:
pixels_to_degrees(360, 57, 41, 1600) 

9.25270847823698